# Final pipeline

Everything that led from the four required detector families to the final submissions: a summary of all models, the data strategies and the inference strategies measured one at a time on the reference RT-DETRv2, the modelling experiments (family tuning, the final RT-DETRv2, the ensemble), a table that separates the three kinds of effect, 3-fold cross-validation of the final recipe, how val relates to the public leaderboard, the final selection, and the commands that reproduce it. All numbers come from `experiments/results.csv` or from the shared evaluator on the saved val predictions of those runs (laptop CPU, no GPU). Plans and decisions in order: `docs/pipeline.md`.

| # | Section |
| --- | --- |
| 1 | [Setup](#1-setup) |
| 2 | [Summary of all models](#2-summary-of-all-models) |
| 3 | [Data strategies](#3-data-strategies) |
| 4 | [Inference strategies](#4-inference-strategies) |
| 5 | [Modelling](#5-modelling) |
| 6 | [Separating the effects](#6-separating-the-effects) |
| 7 | [Cross-validation](#7-cross-validation) |
| 8 | [Validation and the public leaderboard](#8-validation-and-the-public-leaderboard) |
| 9 | [Final selection](#9-final-selection) |
| 10 | [Reproduction](#10-reproduction) |
| 11 | [Conclusions](#11-conclusions) |

## 1. Setup

Imports, data root from `DATA_ROOT`, the annotations of the adopted split, and `experiments/results.csv`. Rules used throughout:

- **Reference model:** `rtdetr-baseline` (RT-DETRv2 R50, COCO weights, frame step 5, 7 epochs), the best family baseline.
- **Seed noise:** two seeds of the same RT-DETRv2 recipe differ by 0.0094 mAP (`rtdetr-lr_scaled` against `rtdetr-lr_scaled_repeat-s7`, `05_query_based.ipynb` §5). A training change counts only above that, or when it repeats with another seed.
- **Val halves:** settings chosen on the saved predictions are also scored on two disjoint halves of the 7 val clips (A: `uav0000086`, `uav0000137`, `uav0000263`, `uav0000339`; B: `uav0000117`, `uav0000182`, `uav0000305`), as in `docs/pipeline.md`. A post-processing gain counts when it holds on both halves.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import seaborn as sns
from decouple import AutoConfig
from IPython.display import display

# make the repository root importable when the kernel starts in notebooks/
PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
# decouple locates .env from the caller file, which is a temp file inside a kernel
config = AutoConfig(search_path=PROJECT_ROOT)

from src.data.annotations import (
    AnnotationTables,
    load_annotations,
    load_clip_split,
    load_split_clips,
    load_test_images,
)
from src.data.classes import CLASS_NAMES
from src.evaluate import DATA_CONFIG_PATH, METRICS_FILE, PREDICTIONS_FILE, RESULTS_PATH, find_run_dir
from src.evaluation.coco_metrics import evaluate_frame_groups
from src.postprocess import evaluate_postprocess_settings
from src.training.experiment import load_experiment_config, merge_config
from src.visualization.plots import plot_grouped_bars, plot_training_history
from src.visualization.style import FAMILY_COLORS, SIZE_BUCKET_COLORS, set_plot_theme
from src.visualization.tables import load_run_histories

set_plot_theme()
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)
pl.Config.set_fmt_str_lengths(70)
DATA_ROOT = Path(config("DATA_ROOT"))
tables = load_annotations(DATA_ROOT, load_split_clips(DATA_CONFIG_PATH))
val_images = tables.images.filter(pl.col("split") == "val")
results = pl.read_csv(RESULTS_PATH).with_columns(family=pl.col("experiment").str.split("-").list.first())
pl.DataFrame({"results rows": [results.height], "val frames": [val_images.height], "val clips": [val_images["clip"].n_unique()]})

In [ ]:
REFERENCE = "rtdetr-baseline"
SINGLE = "rtdetr-infer_recipe_step2_det500"  # best single model: M1 with 500 detections
HEDGE = "postprocess-ens_recipe_frcnn_yolo_yolo1280_smooth"  # ensemble B
FINAL = "postprocess-ens_rtdetr4_frcnn2_yolo2_smooth"  # ensemble D
FINAL_TRAINVAL = "postprocess-ens_trainval_rtdetr4_frcnn2_yolo2_smooth"  # ensemble D', trained on train and val
SEED_NOISE = 0.0094
METRICS = ["map", "map50", "map75", "ap_small", "ap_medium", "ap_large", "map_max_width_1920"]
DELTA_METRICS = ["map", "ap_small", "map_max_width_1920"]
FOLD_A = ["uav0000086", "uav0000137", "uav0000263", "uav0000339"]
clip_halves = {clip: "a" if clip[:10] in FOLD_A else "b" for clip in val_images["clip"].unique().to_list()}
final_config = load_experiment_config(PROJECT_ROOT / "config/postprocess/ens_rtdetr4_frcnn2_yolo2_smooth.toml")
hedge_config = load_experiment_config(PROJECT_ROOT / "config/postprocess/ens_recipe_frcnn_yolo_yolo1280_smooth.toml")
trainval_config = load_experiment_config(PROJECT_ROOT / "config/postprocess/ens_trainval_rtdetr4_frcnn2_yolo2_smooth.toml")
predictions_cache: dict[str, pl.DataFrame] = {}


def run_dir_of(experiment: str) -> Path:
    return PROJECT_ROOT / find_run_dir(RESULTS_PATH, experiment)


def load_val_predictions(experiment: str) -> pl.DataFrame:
    if experiment not in predictions_cache:
        predictions_cache[experiment] = pl.read_parquet(run_dir_of(experiment) / PREDICTIONS_FILE)
    return predictions_cache[experiment]


def load_metrics(experiment: str) -> dict:
    return json.loads((run_dir_of(experiment) / METRICS_FILE).read_text())


def score_members(
    members: list[str], settings: dict[str, dict], split_tables: AnnotationTables, clip_groups: dict[str, str]
) -> pl.DataFrame:
    # fuse the saved val predictions of the members with the settings of D, one row per override
    base = merge_config(
        final_config,
        {"members": members, "weights": [1.0] * len(members), "fusion": "wbf" if len(members) > 1 else "none"},
    )
    member_predictions = [load_val_predictions(m) for m in members]
    table = evaluate_postprocess_settings(split_tables, member_predictions, base, list(settings.values()), clip_groups)
    return table.with_columns(setting=pl.Series(list(settings)))


def compare_runs(runs: dict[str, tuple[str, str]]) -> pl.DataFrame:
    # one row per run with its change, its reference run, and the differences from it
    plan = pl.DataFrame(
        [{"experiment": e, "change": change, "reference": ref} for e, (change, ref) in runs.items()]
    ).with_row_index("order")
    reference = results.select(
        pl.col("experiment").alias("reference"), *(pl.col(m).alias(f"reference_{m}") for m in DELTA_METRICS)
    )
    delta = pl.col("delta_map")
    return (
        plan.join(results.select("experiment", *DELTA_METRICS, "ms_per_image", "gpu_hours"), on="experiment")
        .join(reference, on="reference")
        .sort("order")
        .with_columns((pl.col(m) - pl.col(f"reference_{m}")).round(4).alias(f"delta_{m}") for m in DELTA_METRICS)
        .with_columns(
            verdict=pl.when(delta > SEED_NOISE).then(pl.lit("gain above noise"))
            .when(delta < -SEED_NOISE).then(pl.lit("loss above noise"))
            .otherwise(pl.lit("within noise"))
        )
        .select(
            "experiment", "change", "reference", *DELTA_METRICS,
            *(f"delta_{m}" for m in DELTA_METRICS), "verdict",
            pl.col("ms_per_image").round(1), pl.col("gpu_hours").round(2),
        )
    )

## 2. Summary of all models

The family baselines, the best tuning run of each family, the steps of the final pipeline, and the two final ensembles. Inference time is ms per image on the PC (RTX 5070) at the same input size and batch size 1; for an ensemble it is the sum of its members plus the fusion. GPU hours are training time; for an ensemble the sum of its members' training. The public LB column is filled for the runs that were submitted.

In [ ]:
SUMMARY_RUNS = {
    "faster_rcnn-baseline": "two-stage anchor-based, baseline",
    "faster_rcnn-giou_loss": "two-stage anchor-based, tuned (GIoU loss, 5 epochs)",
    "retinanet-baseline": "one-stage anchor-based, baseline (best of its family)",
    "yolo-baseline": "one-stage anchor-free, baseline (tuning within 0.0004)",
    "rtdetr-baseline": "query-based, baseline (reference)",
    "rtdetr-augment_recipe": "reference + data: recipe augmentation (D1)",
    "rtdetr-recipe_step2": "M1: D1 + frames at step 2 (D2)",
    SINGLE: "M1 + 500 detections (I1), best single model",
    "postprocess-temporal_smooth_rtdetr": "reference + temporal rescoring and smoothing (I6)",
    HEDGE: "ensemble B: 4 members, WBF + I6",
    FINAL: "ensemble D: 8 members, WBF + I6 (final)",
}
summary = (
    pl.DataFrame({"experiment": list(SUMMARY_RUNS), "role": list(SUMMARY_RUNS.values())})
    .with_row_index("order")
    .join(results, on="experiment")
    .sort("order")
    .select(
        "experiment", "role", pl.col(*METRICS).round(4), pl.col("ms_per_image").round(1),
        pl.col("gpu_hours").round(2), pl.col("public_lb").round(4),
    )
)
summary

*Takeaway: written from the outputs.*

In [ ]:
comparable = results.filter(~pl.col("experiment").str.contains("trainval|_fold|_half"))
LABELLED = {
    "faster_rcnn-giou_loss": "Faster R-CNN (tuned)",
    "retinanet-baseline": "RetinaNet",
    "yolo-baseline": "YOLO11s",
    "rtdetr-baseline": "RT-DETRv2 (reference)",
    SINGLE: "M1 + I1",
    HEDGE: "ensemble B",
    FINAL: "ensemble D",
}
fig, ax = plt.subplots(figsize=(9, 5))
for family, color in FAMILY_COLORS.items():
    rows = comparable.filter(pl.col("family") == family)
    ax.scatter(
        rows["ms_per_image"], rows["map"], color=color, s=36, alpha=0.8,
        marker="D" if family == "postprocess" else "o", label=f"{family} ({rows.height} runs)",
    )
for experiment, label in LABELLED.items():
    row = comparable.filter(pl.col("experiment") == experiment).row(0, named=True)
    ax.annotate(label, (row["ms_per_image"], row["map"]), xytext=(5, -10), textcoords="offset points", fontsize=8)
ax.set_xscale("log")
ax.set(
    xlabel="Inference time (ms per image on the PC, log scale)", ylabel="Val mAP@[.5:.95]",
    title=f"Accuracy and speed of all {comparable.height} runs on the adopted val split",
)
ax.legend(fontsize=8, loc="lower right")
fig.tight_layout()

*Takeaway: written from the outputs.*

Per-class AP of the best model of each family, the best single model, and ensemble D. Bus and truck have 2 and few val clips, so their AP is the noisiest (`docs/pipeline.md` §2, observation 7).

In [ ]:
CLASS_RUNS = ["faster_rcnn-giou_loss", "retinanet-baseline", "yolo-baseline", REFERENCE, SINGLE, FINAL]
per_class = pl.DataFrame(
    [{"experiment": e} | load_metrics(e)["evaluation"]["per_class_ap"] for e in CLASS_RUNS]
).with_columns(pl.col(*CLASS_NAMES).round(3))
val_class_counts = tables.objects.filter(pl.col("split") == "val").group_by("class_id").len()
fig, ax = plt.subplots(figsize=(10, 4.2))
sns.heatmap(
    per_class.select(CLASS_NAMES).to_numpy(), annot=True, fmt=".3f", cmap="Blues", vmin=0, vmax=0.6,
    xticklabels=[
        f"{name}\n{val_class_counts.filter(pl.col('class_id') == i)['len'].sum():,} GT"
        for i, name in enumerate(CLASS_NAMES)
    ],
    yticklabels=per_class["experiment"].to_list(), ax=ax,
)
ax.set(xlabel="Class (val GT boxes)", ylabel="", title="Per-class AP@[.5:.95] on val")
fig.tight_layout()

*Takeaway: written from the outputs.*

In [ ]:
SIZE_RUNS = ["faster_rcnn-giou_loss", "retinanet-baseline", "yolo-baseline", REFERENCE, SINGLE, FINAL]
size_table = (
    results.filter(pl.col("experiment").is_in(SIZE_RUNS))
    .unpivot(["ap_small", "ap_medium", "ap_large"], index="experiment", variable_name="size", value_name="ap")
    .with_columns(
        pl.col("size").str.replace("ap_", ""),
        order=pl.col("experiment").replace_strict({e: i for i, e in enumerate(SIZE_RUNS)}),
    )
    .sort("order", "size")
)
fig = plot_grouped_bars(
    size_table, "size", "ap", "experiment",
    ("Object size (COCO bucket)", "AP@[.5:.95]", "AP by object size on val"), "{:.2f}",
)
fig.axes[0].legend(fontsize=7, loc="upper left")

*Takeaway: written from the outputs.*

## 3. Data strategies

Changes to the training data, one at a time on the reference RT-DETRv2 (frame step 5, 7 epochs, about 1.5 GPU hours each) unless the reference column says otherwise. All configs are in `config/rtdetr/` and `config/faster_rcnn/`; the code is `src/data/augment.py` and the `[augment]` and `[ignored_regions]` tables of the baselines.

- **D1 recipe augmentation** (`augment_recipe`): horizontal flip, photometric distortion, multi-scale training (shorter side 832 to 1216 px), and a clean last epoch, after the official RT-DETRv2 recipe ([dataloader config](https://github.com/lyuwenyu/RT-DETR/blob/main/rtdetrv2_pytorch/configs/rtdetrv2/include/dataloader.yml)). `augment_flip_photometric` is the same without multi-scale.
- **D2 frame step 2** (`step2`): 12,240 training frames instead of 4,919, 3 epochs in the same budget.
- **D3 ignored-region masking** (`mask_ignored`): ignored regions filled with the train mean colour, except inside labelled boxes.
- **Low-resolution augmentation** (`recipe_step2_low_res`): a quarter of the augmented frames shrunk 2×, on M1. Motivated by test frame sizes, a deviation from data rule 4 decided before the run (`docs/pipeline.md` §12.5).
- **D1 on Faster R-CNN** (`giou_recipe`): the same augmentation on the tuned Faster R-CNN member.
- **Train and val** (`trainval_*`): the final recipes trained on all 63 clips. Val is then seen in training, so these runs are checked only on the public LB (section 8).

In [ ]:
DATA_RUNS = {
    "rtdetr-augment_flip_photometric": ("D1 without multi-scale", REFERENCE),
    "rtdetr-augment_recipe": ("D1 recipe augmentation", REFERENCE),
    "rtdetr-step2": ("D2 frames at step 2", REFERENCE),
    "rtdetr-mask_ignored": ("D3 ignored regions masked", REFERENCE),
    "rtdetr-recipe_step2": ("D1 + D2 (M1)", REFERENCE),
    "rtdetr-recipe_step2_repeat-s7": ("D1 + D2 (M1), seed 7", REFERENCE),
    "rtdetr-recipe_step2_low_res": ("low-resolution augmentation on M1", "rtdetr-recipe_step2"),
    "faster_rcnn-giou_recipe": ("D1 on the tuned Faster R-CNN", "faster_rcnn-giou_loss"),
}
compare_runs(DATA_RUNS)

*Takeaway: written from the outputs.*

In [ ]:
CURVE_RUNS = [REFERENCE, "rtdetr-augment_flip_photometric", "rtdetr-augment_recipe", "rtdetr-mask_ignored"]
histories = load_run_histories({e: run_dir_of(e) for e in CURVE_RUNS})
fig = plot_training_history(
    histories,
    {"val/map": "Monitoring mAP (val frames at step 5)", "train/loss": "Train loss"},
    "RT-DETRv2 at frame step 5: reference and data strategies",
)

*Takeaway: written from the outputs.*

## 4. Inference strategies

Changes that need no training: each reuses the `best.pt` of its reference run with other inference settings (`config/<family>/infer_*.toml`, `src.predict --inference-config`), or post-processes its saved predictions (`config/postprocess/`, `src/postprocess.py`).

- **I1 500 detections**: RT-DETRv2 keeps up to 500 (query, class) pairs instead of 300, the LB limit.
- **I2 horizontal flip**: predictions on mirrored frames, alone and fused with the plain pass by WBF.
- **I3 inference size**: shorter side 1152, 1280, 1408 px instead of the training size 1024 px; for YOLO `imgsz` 1280 instead of 1824.
- **I6 temporal rescoring and box smoothing** (bonus notebook `07_bonus_temporality.ipynb`): detections linked across consecutive frames, scores moved halfway toward the track maximum, boxes smoothed along the track.

In [ ]:
INFERENCE_RUNS = {
    "rtdetr-infer_det500": ("I1 500 detections", REFERENCE),
    SINGLE: ("I1 500 detections", "rtdetr-recipe_step2"),
    "rtdetr-infer_hflip": ("I2 flipped frames alone", REFERENCE),
    "postprocess-ens_rtdetr_hflip": ("I2 flip TTA (WBF of both passes)", REFERENCE),
    "rtdetr-infer_recipe_hflip": ("I2 flipped frames alone, model trained with flips", "rtdetr-augment_recipe"),
    "yolo-infer_hflip": ("I2 flipped frames alone", "yolo-baseline"),
    "rtdetr-infer_size1152": ("I3 shorter side 1152 px", REFERENCE),
    "rtdetr-infer_size1280": ("I3 shorter side 1280 px", REFERENCE),
    "rtdetr-infer_size1408": ("I3 shorter side 1408 px", REFERENCE),
    "rtdetr-infer_recipe_size1152": ("I3 shorter side 1152 px", "rtdetr-augment_recipe"),
    "rtdetr-infer_recipe_step2_size1152": ("I3 shorter side 1152 px", "rtdetr-recipe_step2"),
    "yolo-infer_imgsz1280": ("I3 imgsz 1280 instead of 1824", "yolo-baseline"),
    "postprocess-temporal_rtdetr": ("I6 temporal rescoring", REFERENCE),
    "postprocess-temporal_smooth_rtdetr": ("I6 temporal rescoring + box smoothing", REFERENCE),
}
compare_runs(INFERENCE_RUNS)

*Takeaway: written from the outputs.*

In [ ]:
SIZE_SWEEP = {1024: REFERENCE, 1152: "rtdetr-infer_size1152", 1280: "rtdetr-infer_size1280", 1408: "rtdetr-infer_size1408"}
sweep = (
    pl.DataFrame({"shorter_side": list(SIZE_SWEEP), "experiment": list(SIZE_SWEEP.values())})
    .join(results, on="experiment")
    .sort("shorter_side")
)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(sweep["shorter_side"], sweep["map"], marker="o", color=FAMILY_COLORS["rtdetr"], label="all sizes")
axes[0].plot(sweep["shorter_side"], sweep["map_max_width_1920"], marker="s", color="0.4", label="frames ≤ 1920 px")
axes[0].set(xlabel="Inference shorter side (px)", ylabel="Val mAP@[.5:.95]", title="mAP")
axes[0].legend(fontsize=8)
for size, color in SIZE_BUCKET_COLORS.items():
    axes[1].plot(sweep["shorter_side"], sweep[f"ap_{size}"], marker="o", color=color, label=size)
axes[1].set(xlabel="Inference shorter side (px)", ylabel="AP@[.5:.95]", title="AP by object size")
axes[1].legend(fontsize=8)
fig.suptitle("Inference size of rtdetr-baseline (trained at 1024 px)")
fig.tight_layout()

*Takeaway: written from the outputs.*

## 5. Modelling

### 5.1 Family tuning and the final RT-DETRv2

The tuning runs of the four required families against their baselines (details in notebooks 02 to 05), the weight EMA, and M1, the final single RT-DETRv2 (D1 and D2 combined, 3 epochs at step 2, best epoch 1), with its seed repeat.

In [ ]:
TUNING_RUNS = {
    "faster_rcnn-lr_cosine": ("cosine LR schedule", "faster_rcnn-baseline"),
    "faster_rcnn-anchors_small": ("anchor sizes 16 to 256", "faster_rcnn-baseline"),
    "faster_rcnn-giou_loss": ("GIoU box loss, 5 epochs", "faster_rcnn-baseline"),
    "faster_rcnn-baseline_5ep-s7": ("5 epochs, seed 7 (noise check)", "faster_rcnn-baseline"),
    "retinanet-focal_loss": ("focal loss gamma 1", "retinanet-baseline"),
    "retinanet-anchors_eda": ("anchors from the box statistics", "retinanet-baseline"),
    "yolo-tal_topk": ("TAL top-k", "yolo-baseline"),
    "yolo-scale_025": ("scale augmentation 0.25", "yolo-baseline"),
    "yolo-imgsz_1280": ("imgsz 1280", "yolo-baseline"),
    "rtdetr-lr_scaled": ("LR scaled to batch 4", REFERENCE),
    "rtdetr-lr_scaled_repeat-s7": ("LR scaled, seed 7", REFERENCE),
    "rtdetr-denoising_groups": ("200 denoising queries", REFERENCE),
    "rtdetr-ema": ("weight EMA", REFERENCE),
    "rtdetr-recipe_step2": ("M1: D1 + D2", REFERENCE),
    "rtdetr-recipe_step2_repeat-s7": ("M1, seed 7", REFERENCE),
}
compare_runs(TUNING_RUNS)

*Takeaway: written from the outputs.*

### 5.2 Ensembles

Weighted Boxes Fusion ([Solovyev et al., 2021](https://arxiv.org/abs/1910.13302); `src/inference/fusion.py`) of the saved predictions of several runs: boxes of all members are clustered per class at IoU 0.65 and each cluster becomes one box with the score-weighted mean coordinates and the mean score over all members. The ensemble runs in `results.csv`, in the order they were built (R `rtdetr-baseline`, F `faster_rcnn-giou_loss`, Y `yolo-baseline`), each against the run it extends:

In [ ]:
ENSEMBLE_RUNS = {
    "postprocess-ens_rtdetr_frcnn": ("WBF R + F", REFERENCE),
    "postprocess-ens_rtdetr_frcnn_yolo": ("WBF R + F + Y", "postprocess-ens_rtdetr_frcnn"),
    "postprocess-ens_all_runs": ("4 RT-DETRv2 runs at half weight + F + Y", "postprocess-ens_rtdetr_frcnn_yolo"),
    "postprocess-ens_rtdetr_frcnn_yolo_temporal": ("R + F + Y, then I6 rescoring", "postprocess-ens_rtdetr_frcnn_yolo"),
    "postprocess-ens_rtdetr_frcnn_yolo_temporal_smooth": ("+ box smoothing", "postprocess-ens_rtdetr_frcnn_yolo_temporal"),
    "postprocess-ens_recipe_frcnn_yolo_smooth": ("R replaced by augment_recipe (D1)", "postprocess-ens_rtdetr_frcnn_yolo_temporal_smooth"),
    HEDGE: ("+ yolo-imgsz_1280 (B)", "postprocess-ens_recipe_frcnn_yolo_smooth"),
    FINAL: ("+ 3 RT-DETRv2 members and faster_rcnn-lr_cosine (D)", HEDGE),
}
compare_runs(ENSEMBLE_RUNS)

*Takeaway: written from the outputs.*

Forward selection from B to D, recomputed here on the saved val predictions with the settings of D (WBF IoU 0.65, equal weights, I6). Each step adds the member that raised val mAP most in the grid of `docs/pipeline.md` §12.2; the last row is the keep check of `faster_rcnn-giou_recipe` (section 3), which replaces `faster_rcnn-giou_loss` in D. Members were chosen on all of val, so the halves are the check.

In [ ]:
b_members = list(hedge_config["members"])
d_members = list(final_config["members"])
SELECTION_STEPS = {
    "B": b_members,
    "B + infer_recipe_step2_det500": b_members + [SINGLE],
    "+ recipe_step2_repeat-s7": b_members + [SINGLE, "rtdetr-recipe_step2_repeat-s7"],
    "+ infer_recipe_hflip (C)": b_members + [SINGLE, "rtdetr-recipe_step2_repeat-s7", "rtdetr-infer_recipe_hflip"],
    "+ faster_rcnn-lr_cosine (D)": d_members,
    "D with giou_recipe instead of giou_loss": [
        "faster_rcnn-giou_recipe" if m == "faster_rcnn-giou_loss" else m for m in d_members
    ],
}
selection = pl.concat(
    [score_members(members, {name: {}}, tables, clip_halves) for name, members in SELECTION_STEPS.items()]
)
selection.select(
    "setting", pl.col("map", "map_a", "map_b", "ap_small", "map_max_width_1920").round(4),
    *(pl.col(c).diff().round(4).alias(f"delta_{c}") for c in ("map", "map_a", "map_b")),
)

*Takeaway: written from the outputs.*

## 6. Separating the effects

The path from the reference to D, one kind of change per step, all scored here on the same val frames and halves: data (training frames and augmentation), inference (500 detections, temporal rescoring and smoothing), and modelling (the ensemble). The 2 × 2 table then crosses the model (single or ensemble) with the temporal step, to show whether the inference gain depends on the model.

In [ ]:
NO_TEMPORAL = {"temporal": {"enabled": False}}
path_rows = pl.concat(
    [
        score_members([REFERENCE], {"reference (rtdetr-baseline)": NO_TEMPORAL}, tables, clip_halves),
        score_members(["rtdetr-recipe_step2"], {"+ data: D1 + D2 (M1)": NO_TEMPORAL}, tables, clip_halves),
        score_members(
            [SINGLE],
            {"+ inference: I1 500 detections": NO_TEMPORAL, "+ inference: I6 temporal and smoothing": {}},
            tables, clip_halves,
        ),
        score_members(d_members, {"ensemble D without I6": NO_TEMPORAL}, tables, clip_halves),
    ]
)
d_row = selection.filter(pl.col("setting") == "+ faster_rcnn-lr_cosine (D)").with_columns(setting=pl.lit("ensemble D"))
effects = pl.concat([path_rows, d_row])
FACTORS = ["reference", "data", "inference", "inference", "modelling", "inference"]
effects.select(
    pl.Series("factor", FACTORS), "setting",
    pl.col("map", "map_a", "map_b", "ap_small", "map_max_width_1920").round(4),
    *(pl.col(c).diff().round(4).alias(f"delta_{c}") for c in ("map", "map_a", "map_b")),
)

In the path above, the step from "+ inference: I6" to "ensemble D without I6" swaps the temporal step for the ensemble, so it is not one factor. The 2 × 2 table isolates each factor at both levels of the other:

In [ ]:
def map_of(setting: str) -> float:
    return effects.filter(pl.col("setting") == setting)["map"].item()


grid = pl.DataFrame(
    {
        "model": ["M1 + I1 (single)", "D members (ensemble of 8)"],
        "without I6": [map_of("+ inference: I1 500 detections"), map_of("ensemble D without I6")],
        "with I6": [map_of("+ inference: I6 temporal and smoothing"), map_of("ensemble D")],
    }
).with_columns(i6_gain=pl.col("with I6") - pl.col("without I6"))
display(grid.with_columns(pl.selectors.float().round(4)))
pl.DataFrame(
    {
        "effect": [
            "data (D1 + D2) on the reference",
            "I6 on the single model",
            "I6 on the ensemble",
            "ensemble without I6",
            "ensemble with I6",
        ],
        "delta_map": [
            map_of("+ data: D1 + D2 (M1)") - map_of("reference (rtdetr-baseline)"),
            grid["i6_gain"][0],
            grid["i6_gain"][1],
            grid["without I6"][1] - grid["without I6"][0],
            grid["with I6"][1] - grid["with I6"][0],
        ],
    }
).with_columns(pl.col("delta_map").round(4))

*Takeaway: written from the outputs.*

## 7. Cross-validation

3-fold `GroupKFold` by clip over all 63 labelled clips (shuffled, seed 42, 21 clips per fold, `[folds]` in `config/data.toml`), for the two members that the time allowed: the final RT-DETRv2 recipe (1 epoch at step 2 with 500 detections, `config/rtdetr/cv_recipe_step2_fold<k>.toml`) and the Faster R-CNN GIoU member (2 epochs at step 5, `config/faster_rcnn/cv_giou_loss_fold<k>.toml`), plus their WBF with the settings of D (`config/postprocess/cv_ens_rtdetr_frcnn_smooth_fold<k>.toml`). Each fold trains on 42 clips and validates on the other 21, so fold numbers are compared with each other, not with the adopted val split.

In [ ]:
cv = (
    results.filter(pl.col("experiment").str.contains(r"_fold\d$"))
    .with_columns(
        fold=pl.col("experiment").str.extract(r"fold(\d)$").cast(pl.Int64),
        model=pl.col("experiment").str.replace(r"_fold\d$", ""),
    )
)
fold_sizes = pl.DataFrame(
    [
        {"fold": k} | {key: load_metrics(f"rtdetr-cv_recipe_step2_fold{k}")["evaluation"][key] for key in ("frames", "objects")}
        for k in range(3)
    ]
)
cv_table = (
    cv.pivot("model", index="fold", values="map")
    .join(fold_sizes, on="fold")
    .with_columns(ensemble_gain=pl.col("postprocess-cv_ens_rtdetr_frcnn_smooth") - pl.col("rtdetr-cv_recipe_step2"))
    .sort("fold")
)
display(cv_table.with_columns(pl.selectors.float().round(4)))
cv.group_by("model", maintain_order=True).agg(
    pl.col("map").mean().round(4).alias("mean_map"),
    pl.col("map").std().round(4).alias("std_map"),
    pl.col("map").min().round(4).alias("min_map"),
    pl.col("map").max().round(4).alias("max_map"),
    pl.col("ap_small").mean().round(4).alias("mean_ap_small"),
    pl.col("map_max_width_1920").mean().round(4).alias("mean_map_max_width_1920"),
)

*Takeaway: written from the outputs.*

In [ ]:
fig = plot_grouped_bars(
    cv.with_columns(pl.format("fold {}", "fold").alias("fold_name")).sort("fold", "model"),
    "fold_name", "map", "model",
    ("Cross-validation fold (21 val clips each)", "Val mAP@[.5:.95]", "3-fold cross-validation of the final recipe"),
    "{:.3f}",
)
fig.axes[0].legend(fontsize=8, loc="lower right")

*Takeaway: written from the outputs.*

The fold ensembles include the temporal step, so their gain over RT-DETRv2 alone mixes fusion and I6. Both parts are separated here on each fold from the saved fold predictions, and the same two-member ensemble is scored on the adopted val split for comparison.

In [ ]:
fold_rows = []
for k in range(3):
    fold_tables = load_annotations(DATA_ROOT, load_clip_split(DATA_CONFIG_PATH, f"fold_{k}"))
    fold_clips = {clip: "fold" for clip in fold_tables.images.filter(pl.col("split") == "val")["clip"].unique().to_list()}
    rtdetr, frcnn = f"rtdetr-cv_recipe_step2_fold{k}", f"faster_rcnn-cv_giou_loss_fold{k}"
    fold_rows.append(
        pl.concat(
            [
                score_members([rtdetr], {"RT-DETRv2": NO_TEMPORAL, "RT-DETRv2 + I6": {}}, fold_tables, fold_clips),
                score_members([rtdetr, frcnn], {"WBF": NO_TEMPORAL, "WBF + I6": {}}, fold_tables, fold_clips),
            ]
        ).select(pl.lit(f"fold {k}").alias("split"), "setting", "map")
    )
main_rows = pl.concat(
    [
        score_members([SINGLE], {"RT-DETRv2": NO_TEMPORAL, "RT-DETRv2 + I6": {}}, tables, clip_halves),
        score_members([SINGLE, "faster_rcnn-giou_loss"], {"WBF": NO_TEMPORAL, "WBF + I6": {}}, tables, clip_halves),
    ]
).select(pl.lit("adopted val").alias("split"), "setting", "map")
separation = (
    pl.concat([*fold_rows, main_rows])
    .pivot("setting", index="split", values="map")
    .with_columns(
        i6_on_single=pl.col("RT-DETRv2 + I6") - pl.col("RT-DETRv2"),
        fusion_without_i6=pl.col("WBF") - pl.col("RT-DETRv2"),
        i6_on_fusion=pl.col("WBF + I6") - pl.col("WBF"),
    )
)
separation.with_columns(pl.selectors.float().round(4))

*Takeaway: written from the outputs.*

## 8. Validation and the public leaderboard

Every submission so far, with its val mAP and public LB score. The LB was used only as a final check of models chosen on val (data rule 4). The train and val models have no comparable val score: their val clips were in training.

In [ ]:
submitted = (
    results.filter(pl.col("public_lb").is_not_null())
    .with_columns(is_val_comparable=~pl.col("experiment").str.contains("trainval"))
    .sort("public_lb", descending=True)
)
lb_table = submitted.select(
    "experiment", "is_val_comparable", pl.col("map", "map_max_width_1920", "public_lb").round(4),
    (pl.col("map") - pl.col("public_lb")).round(4).alias("val_minus_lb"),
)
display(lb_table)
lb_comparable = submitted.filter(pl.col("is_val_comparable"))
lb_comparable.select(
    pl.len().alias("submissions"),
    pl.corr("map", "public_lb", method="spearman").round(3).alias("spearman_val"),
    pl.corr("map_max_width_1920", "public_lb", method="spearman").round(3).alias("spearman_val_max_1920"),
    pl.corr("map", "public_lb").round(3).alias("pearson_val"),
)

*Takeaway: written from the outputs.*

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for family, color in FAMILY_COLORS.items():
    rows = lb_comparable.filter(pl.col("family") == family)
    ax.scatter(rows["map"], rows["public_lb"], color=color, s=40, label=family, marker="D" if family == "postprocess" else "o")
for row in lb_comparable.iter_rows(named=True):
    ax.annotate(row["experiment"].split("-", 1)[1], (row["map"], row["public_lb"]), xytext=(4, -9), textcoords="offset points", fontsize=7)
ax.set(xlabel="Val mAP@[.5:.95] (adopted split)", ylabel="Public LB score", title="Val against public LB, submissions with a comparable val score")
ax.legend(fontsize=8)
fig.tight_layout()

*Takeaway: written from the outputs.*

Val frames come in three widths, while test has no frame above 1920 px and 16% half-resolution frames (test metadata only; nothing is fitted on test). Val mAP per width group of every comparable submission, and how well each group ranks the submissions like the LB:

In [ ]:
test_images = load_test_images(DATA_ROOT)
width_group = (
    pl.when(pl.col("width") < 1000).then(pl.lit("< 1000 px"))
    .when(pl.col("width") < 1400).then(pl.lit("1344 to 1360 px"))
    .when(pl.col("width") <= 1920).then(pl.lit("1904 to 1920 px"))
    .otherwise(pl.lit("2688 px and above"))
)
pl.concat([val_images.select("split", "width"), test_images.select("split", "width")]).group_by(
    "split", width_group.alias("width_group")
).len("frames").with_columns(share=(pl.col("frames") / pl.col("frames").sum().over("split")).round(3)).pivot(
    "split", index="width_group", values=["frames", "share"]
).sort("width_group")

*Takeaway: written from the outputs.*

In [ ]:
val_width_groups = val_images.select("image_id", width_group.alias("group"))
width_rows = []
for experiment in lb_comparable["experiment"]:
    groups = evaluate_frame_groups(tables, "val", load_val_predictions(experiment), val_width_groups)
    width_rows.append(groups.select(pl.lit(experiment).alias("experiment"), "group", "map"))
by_width = (
    pl.concat(width_rows)
    .pivot("group", index="experiment", values="map")
    .join(lb_comparable.select("experiment", "map", "public_lb"), on="experiment")
    .sort("public_lb", descending=True)
)
display(by_width.with_columns(pl.selectors.float().round(4)))
width_columns = [c for c in by_width.columns if c.endswith(" px") or c.endswith("above")]
by_width.select(
    *(pl.corr(c, "public_lb", method="spearman").round(3).alias(c) for c in [*width_columns, "map"])
).with_columns(pl.lit("Spearman with LB").alias("statistic"))

*Takeaway: written from the outputs.*

## 9. Final selection

The rule was fixed on val before any LB score of the candidates (`docs/pipeline.md` §12.3, §12.7):

1. **D** (`ens_rtdetr4_frcnn2_yolo2_smooth`), the best validated pipeline: highest val mAP and higher than B on both val halves.
2. **D′** (`ens_trainval_rtdetr4_frcnn2_yolo2_smooth`), the same recipe with the RT-DETRv2 and Faster R-CNN members retrained on all 63 clips (same epochs as the best epoch of their val runs; YOLO stays train only because its best epoch falls under a decaying LR that a shorter run does not reproduce). It cannot be validated, so it is the second pick rather than the first. Fallback fixed in advance: D and B, if D′ scored below B on the LB, as a sign of a defect.

In [ ]:
CANDIDATES = {FINAL: "D", FINAL_TRAINVAL: "D′", HEDGE: "B", SINGLE: "best single model"}
display(
    submitted.filter(pl.col("experiment").is_in(list(CANDIDATES)))
    .with_columns(name=pl.col("experiment").replace_strict(CANDIDATES))
    .select("name", "experiment", "is_val_comparable", pl.col("map", "map_max_width_1920", "public_lb").round(4), pl.col("ms_per_image").round(1), pl.col("gpu_hours").round(2))
)
pl.DataFrame({"member of D": d_members, "member of D′": list(trainval_config["members"])}).join(
    results.select(pl.col("experiment").alias("member of D"), "family", "map", "ap_small", pl.col("ms_per_image").round(1), pl.col("gpu_hours").round(2)),
    on="member of D",
    maintain_order="left",
)

*Takeaway: written from the outputs.*

## 10. Reproduction

Environment (Python 3.12, `uv`, exact versions in `uv.lock`; on Kaggle `requirements-kaggle.txt` over the preinstalled `torch`):

```bash
uv sync
cp .env.example .env   # set DATA_ROOT to the competition data
```

Each queue item trains (or predicts, or fuses), predicts all val frames and the test frames, evaluates with the shared evaluator, validates the CSV, and appends the `results.csv` row (`scripts/run_queue.sh`, `scripts/finish_run.sh`). Seeds: 42 everywhere except the seed 7 repeats (`*_repeat`, suffix `-s7`). GPU training is not bit-exact, so reproduced runs match within the seed noise rather than exactly.

**D** (members in the order of the config; an `infer` item needs its source run, trained earlier in the queue):

```bash
scripts/run_queue.sh \
    train:config/rtdetr/augment_recipe.toml \
    infer:config/rtdetr/infer_recipe_hflip.toml \
    train:config/rtdetr/recipe_step2.toml \
    infer:config/rtdetr/infer_recipe_step2_det500.toml \
    train:config/rtdetr/recipe_step2_repeat.toml \
    train:config/faster_rcnn/giou_loss.toml \
    train:config/faster_rcnn/lr_cosine.toml \
    train:config/yolo/baseline.toml \
    train:config/yolo/imgsz_1280.toml \
    post:config/postprocess/ens_rtdetr4_frcnn2_yolo2_smooth.toml
```

**D′** (after D, which provides the two YOLO members):

```bash
scripts/run_queue.sh \
    train:config/rtdetr/trainval_recipe_step5.toml \
    infer:config/rtdetr/infer_trainval_recipe_step5_hflip.toml \
    train:config/rtdetr/trainval_recipe_step2.toml \
    infer:config/rtdetr/infer_trainval_recipe_step2_det500.toml \
    train:config/rtdetr/trainval_recipe_step2_repeat.toml \
    train:config/faster_rcnn/trainval_giou_loss.toml \
    train:config/faster_rcnn/trainval_baseline_2ep.toml \
    post:config/postprocess/ens_trainval_rtdetr4_frcnn2_yolo2_smooth.toml
```

**Cross-validation** (section 7), for k in 0, 1, 2:

```bash
scripts/run_queue.sh \
    train:config/rtdetr/cv_recipe_step2_fold<k>.toml \
    train:config/faster_rcnn/cv_giou_loss_fold<k>.toml \
    post:config/postprocess/cv_ens_rtdetr_frcnn_smooth_fold<k>.toml
```

**Submission:** check the CSV, then copy it to `submissions/` and upload:

```bash
uv run python -m src.submission.validate runs/<experiment>/<timestamp>/predictions.csv
scripts/finish_run.sh runs/<experiment>/<timestamp> --submit
```

The recorded runs of the final members (config, seed, commit, training budget):

In [ ]:
reproduction_members = [*d_members, *[m for m in trainval_config["members"] if m not in d_members]]
pl.DataFrame({"experiment": reproduction_members}).join(
    results.select(
        "experiment", "config", "seed", pl.col("commit").str.slice(0, 7), "is_dirty", "frame_step", "epochs",
        "effective_batch_size", pl.col("gpu_hours").round(2),
    ),
    on="experiment",
    maintain_order="left",
)

*Takeaway: written from the outputs.*

## 11. Conclusions

*Takeaway: written from the outputs.*